# Import Libraries

In [34]:
import boto3
import pandas as pd
import numpy as np
import sklearn
import sagemaker
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import TruncatedSVD
import re
import nltk
import time
from botocore.exceptions import BotoCoreError, ClientError
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

# Checking Up S3 Buckets and Locate Datasets

In [35]:
s3_resource = boto3.resource('s3')
bucket = s3_resource.Bucket('ads508-team1-sephora')

for items in bucket.objects.all():
    print(items.key)

Model/
Model/final_features/
Model/final_features/df_product_final.parquet
Model/final_features/reviews_tfidf_df.parquet
Model/final_splits/X_test_love.parquet
Model/final_splits/X_test_rating.parquet
Model/final_splits/X_train_love.parquet
Model/final_splits/X_train_rating.parquet
Model/final_splits/X_val_love.parquet
Model/final_splits/X_val_rating.parquet
Model/final_splits/product_X_test.parquet
Model/final_splits/product_X_train.parquet
Model/final_splits/product_X_val.parquet
Model/final_splits/product_y_test.parquet
Model/final_splits/product_y_train.parquet
Model/final_splits/product_y_val.parquet
Model/final_splits/review_X_test.parquet
Model/final_splits/review_X_train.parquet
Model/final_splits/review_X_val.parquet
Model/final_splits/review_y_test.parquet
Model/final_splits/review_y_train.parquet
Model/final_splits/review_y_val.parquet
Model/final_splits/y_test_love.parquet
Model/final_splits/y_test_rating.parquet
Model/final_splits/y_train_love.parquet
Model/final_splits/y_

# Load the Product Info Dataset

In [36]:
# Define the S3 path
s3_path = 's3://ads508-team1-sephora/curated/products/products.parquet'
s3_path_reviews = 's3://ads508-team1-sephora/curated/reviews/reviews.parquet'

# Load the parquet directly into a DataFrame
df_products = pd.read_parquet(s3_path)
df_reviews = pd.read_parquet(s3_path_reviews)

# Display the first few rows
df_products.head().T

,0,1,2,3,4
product_id,P473671,P473668,P473662,P473660,P473658
product_name,Fragrance Discovery Set,La Habana Eau de Parfum,Rainbow Bar Eau de Parfum,Kasbah Eau de Parfum,Purple Haze Eau de Parfum
brand_id,6342,6342,6342,6342,6342
brand_name,19-69,19-69,19-69,19-69,19-69
loves_count,6320,3827,3253,3018,2691
rating,3.6364,4.1538,4.25,4.4762,3.2308
reviews,11.0,13.0,16.0,21.0,13.0
size,None,3.4 oz/ 100 mL,3.4 oz/ 100 mL,3.4 oz/ 100 mL,3.4 oz/ 100 mL
variation_type,None,Size + Concentration + Formulation,Size + Concentration + Formulation,Size + Concentration + Formulation,Size + Concentration + Formulation
variation_value,None,3.4 oz/ 100 mL,3.4 oz/ 100 mL,3.4 oz/ 100 mL,3.4 oz/ 100 mL


In [37]:
# Product file shape
df_products.shape

(8494, 27)

# Project Will be Focus on Makeup Category

In [38]:
# Filter to only focus on the Makeup Category, since we only work on Makeup Category in our company
# Filter the dataframe to only include the Makeup category
df_products = df_products[df_products['primary_category'] == 'Makeup'].copy()

# Reset the index since we've removed rows from other categories
df_products = df_products.reset_index(drop=True)

# Quick check of the new dataset distribution
print(f"Filtered Shape: {df_products.shape}")
print("\nBreakdown of Makeup Sub-categories:")
print(df_products['secondary_category'].value_counts())

# Display the first few rows
df_products.head()

Filtered Shape: (2369, 27)

Breakdown of Makeup Sub-categories:
secondary_category
Eye                      711
Face                     659
Lip                      411
Brushes & Applicators    233
Cheek                    165
Nail                      52
Accessories               45
Mini Size                 37
Value & Gift Sets         36
Makeup Palettes           20
Name: count, dtype: int64


,product_id,product_name,brand_id,brand_name,loves_count,rating,reviews,size,variation_type,variation_value,...,online_only,out_of_stock,sephora_exclusive,highlights,primary_category,secondary_category,tertiary_category,child_count,child_max_price,child_min_price
0,P398965,Rose Lip Conditioner,7054,AERIN,20562,4.2600,527.0,0.34 oz/ 10 mL,Scent,1 Rose,...,1,0,0,"['Hydrating', 'Good for: Dryness']",Makeup,Lip,Lip Balm & Treatment,0,NaN,NaN
1,P503741,Lip Treatment Oil,7103,Ami Colé,22871,4.7053,95.0,0.15 oz / 4.5 ml,Color,Excellence,...,0,0,1,"['Vegan', 'High Shine Finish', 'Clean at Sephora', 'Hydrating', 'Black Owned at Sephora', 'Cruelty-Free']",Makeup,Lip,Lip Balm & Treatment,3,20.0,20.0
2,P503754,Skin-Enhancing Tinted Moisturizer,7103,Ami Colé,6596,4.8966,58.0,1 oz / 30 ml,Color,Rich 2,...,0,0,1,"['Vegan', 'Natural Finish', 'Clean at Sephora', 'Medium Coverage', 'Black Owned at Sephora', 'Cruelty-Free']",Makeup,Face,Tinted Moisturizer,5,32.0,32.0
3,P503762,Lash-Amplifying Volumizing & Lengthening Mascara,7103,Ami Colé,5015,4.1842,38.0,0.3 oz / 9 ml,Color,Rich Black,...,0,0,1,"['Vegan', 'allure 2022 Best of Beauty Award Winner', 'Clean at Sephora', 'Black Owned at Sephora', 'Cruelty-Free']",Makeup,Eye,Mascara,0,NaN,NaN
4,P503732,Skin Melt Talc-Free Loose Setting Powder,7103,Ami Colé,4978,4.5714,21.0,0. 29 oz / 8.6 g,Color,Rich / Deep,...,1,0,1,"['Vegan', 'Loose Powder Formula', 'Clean at Sephora', 'Matte Finish', 'Black Owned at Sephora', 'Cruelty-Free']",Makeup,Face,Setting Spray & Powder,2,22.0,22.0


# Checking Out the Product Dataset Information after filtering

In [39]:
# Product File Information
df_products.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2369 entries, 0 to 2368
Data columns (total 27 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   product_id          2369 non-null   object 
 1   product_name        2369 non-null   object 
 2   brand_id            2369 non-null   int64  
 3   brand_name          2369 non-null   object 
 4   loves_count         2369 non-null   int64  
 5   rating              2328 non-null   float64
 6   reviews             2328 non-null   float64
 7   size                1667 non-null   object 
 8   variation_type      1851 non-null   object 
 9   variation_value     1802 non-null   object 
 10  variation_desc      1186 non-null   object 
 11  ingredients         2033 non-null   object 
 12  price_usd           2369 non-null   float64
 13  value_price_usd     104 non-null    float64
 14  sale_price_usd      143 non-null    float64
 15  limited_edition     2369 non-null   int64  
 16  new   

In [40]:
# Profuct file description
df_products.describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
product_id,2369,2369,P505461,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
product_name,2369,2360,Makeup Match Powder Brush,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
brand_id,2369.0,NaN,NaN,NaN,5120.040523,1683.804898,1063.0,3976.0,5879.0,6236.0,8000.0
brand_name,2369,115,SEPHORA COLLECTION,188,NaN,NaN,NaN,NaN,NaN,NaN,NaN
loves_count,2369.0,NaN,NaN,NaN,54235.494301,100425.225843,0.0,8304.0,22387.0,53470.0,1401068.0
rating,2328.0,NaN,NaN,NaN,4.146845,0.513878,1.0,3.9286,4.2523,4.4915,5.0
reviews,2328.0,NaN,NaN,NaN,681.998282,1608.242011,1.0,44.0,209.0,625.0,21281.0
size,1667,978,1 oz/ 30 mL,66,NaN,NaN,NaN,NaN,NaN,NaN,NaN
variation_type,1851,4,Color,1486,NaN,NaN,NaN,NaN,NaN,NaN,NaN
variation_value,1802,1352,Black,66,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [41]:
# Checking missing data points
df_products.isnull().sum()

product_id               0
product_name             0
brand_id                 0
brand_name               0
loves_count              0
rating                  41
reviews                 41
size                   702
variation_type         518
variation_value        567
variation_desc        1183
ingredients            336
price_usd                0
value_price_usd       2265
sale_price_usd        2226
limited_edition          0
new                      0
online_only              0
out_of_stock             0
sephora_exclusive        0
highlights             738
primary_category         0
secondary_category       0
tertiary_category      153
child_count              0
child_max_price       1069
child_min_price       1069
dtype: int64

In [42]:
# Checking missing data points percentage
df_products.isnull().mean()*100

product_id             0.000000
product_name           0.000000
brand_id               0.000000
brand_name             0.000000
loves_count            0.000000
rating                 1.730688
reviews                1.730688
size                  29.632756
variation_type        21.865766
variation_value       23.934149
variation_desc        49.936682
ingredients           14.183200
price_usd              0.000000
value_price_usd       95.609962
sale_price_usd        93.963698
limited_edition        0.000000
new                    0.000000
online_only            0.000000
out_of_stock           0.000000
sephora_exclusive      0.000000
highlights            31.152385
primary_category       0.000000
secondary_category     0.000000
tertiary_category      6.458421
child_count            0.000000
child_max_price       45.124525
child_min_price       45.124525
dtype: float64

# Imputation of Missing Value

## 1) Fill in mean rating for missing rating

In [43]:
# Fill in mean Rating for missing Rating
# Calculate the mean of the rating column
mean_rating = df_products['rating'].mean()

# Fill missing values in 'rating' with the mean
df_products['rating'] = df_products['rating'].fillna(mean_rating)

# Verify that there are no more missing values in 'rating'
print(f"Missing ratings remaining: {df_products['rating'].isnull().sum()}")

Missing ratings remaining: 0


## 2) Fill in zero for missing reviews

In [44]:
# Fill in zero for missing reviews
# Fill missing values in 'reviews' with 0
df_products['reviews'] = df_products['reviews'].fillna(0)

# Verify that there are no more missing values in 'reviews'
print(f"Missing reviews remaining: {df_products['reviews'].isnull().sum()}")

Missing reviews remaining: 0


## 3) Fill in NA for missing size, variation_type, variation_value, variation_desc, ingredients, highlights, secondary_category, tertiary_category

In [45]:
# Fill in NA for missing size, variation_type, variation_value, variation_desc, ingredients, highlights, secondary_category, tertiary_category
# List of columns to fill with "NA"
cols_to_fix = [
    'size', 'variation_type', 'variation_value', 
    'variation_desc', 'ingredients', 'highlights', 'secondary_category', 'tertiary_category'
]

# Fill missing values in these columns with "NA"
df_products[cols_to_fix] = df_products[cols_to_fix].fillna("NA")

# Verify that no missing values remain in these columns
print(df_products[cols_to_fix].isnull().sum())

size                  0
variation_type        0
variation_value       0
variation_desc        0
ingredients           0
highlights            0
secondary_category    0
tertiary_category     0
dtype: int64


## 4) Fill in price_usd for missing value in value_price_usd and sale_price_usd

In [46]:
# Fill in price_usd for missing value in value_price_usd and sale_price_usd
# Fill missing value_price_usd with the standard price_usd
df_products['value_price_usd'] = df_products['value_price_usd'].fillna(df_products['price_usd'])

# Fill missing sale_price_usd with the standard price_usd
df_products['sale_price_usd'] = df_products['sale_price_usd'].fillna(df_products['price_usd'])

# Verify that no missing values remain in these price columns
print(f"Missing value_price_usd: {df_products['value_price_usd'].isnull().sum()}")
print(f"Missing sale_price_usd: {df_products['sale_price_usd'].isnull().sum()}")

Missing value_price_usd: 0
Missing sale_price_usd: 0


## 5) Fill in zero for missing child_max_price and child_min_price

In [47]:
# Fill in zero for missing child_max_price and child_min_price
# Fill missing values for child price range columns with 0
df_products['child_max_price'] = df_products['child_max_price'].fillna(0)
df_products['child_min_price'] = df_products['child_min_price'].fillna(0)

# Verify the changes
print(f"Missing child_max_price: {df_products['child_max_price'].isnull().sum()}")
print(f"Missing child_min_price: {df_products['child_min_price'].isnull().sum()}")

Missing child_max_price: 0
Missing child_min_price: 0


## 6) Confirmation for no missing values

In [48]:
# Checking missing data points percentage
df_products.isnull().mean()*100

product_id            0.0
product_name          0.0
brand_id              0.0
brand_name            0.0
loves_count           0.0
rating                0.0
reviews               0.0
size                  0.0
variation_type        0.0
variation_value       0.0
variation_desc        0.0
ingredients           0.0
price_usd             0.0
value_price_usd       0.0
sale_price_usd        0.0
limited_edition       0.0
new                   0.0
online_only           0.0
out_of_stock          0.0
sephora_exclusive     0.0
highlights            0.0
primary_category      0.0
secondary_category    0.0
tertiary_category     0.0
child_count           0.0
child_max_price       0.0
child_min_price       0.0
dtype: float64

# Feature Engineering

## 1) Initial price feature engineering

In [49]:
# 1. Log transformation of price to handle skewness
df_products['price_log'] = np.log1p(df_products['price_usd'])

# 2. Identify if a product is actually on sale 
# (Comparing sale_price_usd to the original price_usd)
df_products['is_on_sale'] = (df_products['sale_price_usd'] < df_products['price_usd']).astype(int)

# 3. Calculate the discount percentage
# If no sale, this will result in 0.0
df_products['discount_pct'] = (df_products['price_usd'] - df_products['sale_price_usd']) / df_products['price_usd']

# 4. Calculate the value ratio (Value for money)
# (value_price_usd / price_usd)
df_products['value_ratio'] = df_products['value_price_usd'] / df_products['price_usd']

# Display the new features for the first few rows
df_products[['price_usd', 'price_log','sale_price_usd', 'is_on_sale', 'discount_pct', 'value_ratio']].head()

,price_usd,price_log,sale_price_usd,is_on_sale,discount_pct,value_ratio
0,35.0,3.583519,35.0,0,0.0,1.0
1,20.0,3.044522,20.0,0,0.0,1.0
2,32.0,3.496508,32.0,0,0.0,1.0
3,19.0,2.995732,19.0,0,0.0,1.0
4,22.0,3.135494,22.0,0,0.0,1.0


Log transformation of price_usd (price_log): Applied np.log1p to normalize the skewed price distribution and reduce the impact of extreme outliers (like the $1,900 items).

Price sensitivity & promotion tracking (is_on_sale & discount_pct): Created binary indicators and percentage calculations to capture whether a product is discounted relative to its original price.

Perceived value mapping (value_ratio): Calculated the ratio of value_price_usd to price_usd to quantify "value for money," especially for gift sets and bundles.

## 2) Price Bucketing (Business-friendly)

In [50]:
# Define price segments based on the distribution of price_usd
# New bins: 0-25, 25-50, 50-100, 100+
bins = [0, 25, 50, 100, np.inf]
labels = ['budget', 'low_mid', 'mid', 'premium']

# Create the price_bucket feature
df_products['price_bucket'] = pd.cut(df_products['price_usd'], bins=bins, labels=labels)

# Verify the distribution of the new buckets
print(df_products['price_bucket'].value_counts())

price_bucket
low_mid    1239
budget      895
mid         212
premium      23
Name: count, dtype: int64


Easier for Models & Business Interpretation: Converting exact prices into ranges reduces noise for machine learning algorithms and aligns with how retail stakeholders view product "tiers."

Hypothesis Testing: This allows us to move beyond individual price points and answer high-level business questions, such as: "Do premium products ($100–$200) maintain higher average ratings or 'loves' counts than budget items?"

## 3) Brand Features

In [51]:
# Identify the top 20 brands by product count
top_brands = df_products['brand_name'].value_counts().nlargest(20).index

# Group any brand not in the top 20 as 'Other' to reduce category cardinality
df_products['brand_group'] = df_products['brand_name'].apply(lambda x: x if x in top_brands else 'Other')

# Check the distribution of the new brand groups
print(df_products['brand_group'].value_counts())

brand_group
Other                      1082
SEPHORA COLLECTION          188
tarte                       100
Anastasia Beverly Hills      88
Charlotte Tilbury            76
Fenty Beauty by Rihanna      74
Hourglass                    67
MAKE UP FOR EVER             67
NARS                         59
CLINIQUE                     59
Dior                         56
HUDA BEAUTY                  54
Too Faced                    53
Urban Decay                  50
Benefit Cosmetics            49
PAT McGRATH LABS             49
Laura Mercier                47
Bobbi Brown                  39
Natasha Denona               38
Smashbox                     38
MILK MAKEUP                  36
Name: count, dtype: int64


Brand Categorization (brand_group): We identified the top 20 brands by product count and consolidated all remaining brands into a single "Other" category.

Solves High-Cardinality: Prevents "noise" in machine learning models by reducing hundreds of unique brand names—many with only 1 or 2 products—into a manageable set of features.

Captures Brand Reputation: Isolates the "Power Players" (e.g., SEPHORA COLLECTION, Clinique, Dior) where we have enough data to draw statistically significant conclusions about brand performance.

Dimensionality Reduction: Streamlines future visualizations and one-hot encoding, ensuring that our analysis (if we need to) focuses on the most influential market competitors rather than long-tail outliers.

## 4) One of the Outcome/Target - Popularity Features

In [52]:
# 1. Log transformation of loves_count
# Helps manage the massive range (0 to 1.4M+) and reduces skewness
df_products['log_loves'] = np.log1p(df_products['loves_count'])

# 2. Loves per Price (Value Sentiment)
# Measures how much 'love' a product gets per dollar spent
df_products['loves_per_price'] = df_products['loves_count'] / df_products['price_usd']

# 3. Popularity Score (Interaction weighted by Rating)
# A simple way to combine volume (reviews) and sentiment (rating)
df_products['popularity_index'] = df_products['reviews'] * df_products['rating']

# Display the results for high-interaction products
df_products[['product_name', 'loves_count', 'log_loves', 'loves_per_price', 'popularity_index']].sort_values(by='loves_count', ascending=False).head()

,product_name,loves_count,log_loves,loves_per_price,popularity_index
1645,Soft Pinch Liquid Blush,1401068,14.152746,60916.000000,21466.9948
1404,Radiant Creamy Concealer,1153594,13.958394,36049.812500,55517.1960
1742,Cream Lip Stain Liquid Lipstick,1029051,13.844149,68603.400000,48000.6311
550,Gloss Bomb Universal Lip Luminizer,968317,13.783316,46110.333333,56258.8552
551,Pro Filt’r Soft Matte Longwear Liquid Foundation,856497,13.660607,21412.425000,68342.8860


Popularity & Engagement Metrics (log_loves, loves_per_price, popularity_index): We engineered several features to quantify product "hype" and consumer sentiment.

Captures Customer Demand: By combining loves_count and reviews, we create a proxy for market demand that distinguishes between products people simply "wishlist" versus those they actually purchase and evaluate.

Addresses Popularity Bias: Applying a log transformation to loves_count (which ranges from 0 to over 1.4M) prevents "viral" outliers—like the Soft Pinch Liquid Blush—from skewing the model's ability to learn from standard products.

Identifies "Cult Favorites": The popularity_index (Reviews × Rating) helps the model prioritize products with high-volume, high-quality feedback, effectively filtering out 5-star items that only have a single review.

## 5) Ingredient Features (Text → Structured)

In [53]:
# Top 100 Ingredients
ingredients_long = (
    df_products[['ingredients']]
    .dropna()
    .assign(ingredient=lambda x: x['ingredients'].str.split(','))
    .explode('ingredient')
    .copy()
)

# keep original product index so deduping happens within product
ingredients_long['product_idx'] = ingredients_long.index

ingredients_long['ingredient'] = (
    ingredients_long['ingredient']
    .astype(str)
    .str.lower()
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)                  # normalize spaces
    .str.replace(r"^\[+", "", regex=True)                  # remove leading [
    .str.replace(r"^'+", "", regex=True)                   # remove leading '
    .str.replace(r"^\[\s*\+/-\s*", "", regex=True)         # remove leading [+/-
    .str.replace(r"^\+/-\s*", "", regex=True)              # remove leading +/-
    .str.replace(r"\.+$", "", regex=True)                  # remove trailing periods
    .str.replace(r"\)+$", "", regex=True)                  # remove trailing )
    .str.replace(r"\(.*", "", regex=True)                  # remove anything after (
    .str.replace(r"\s*[\\/]\s*", "/", regex=True)          # normalize slash spacing
    .str.strip()
)

ingredients_long = ingredients_long[
    ~ingredients_long['ingredient'].isin(['', 'na', '1', 'none', 'nan', '+/-'])
].copy()

In [54]:
# Mapping dictionary (used ChatGPT based on top 100 ingredients)
ingredient_map = {
    # water
    'aqua': 'water',
    'eau': 'water',
    'water': 'water',
    'water/aqua': 'water',
    'aqua/water': 'water',
    'aqua/water/eau': 'water',
    'water/aqua/eau': 'water',
    'water//aqua//eau': 'water',
    'water\\aqua\\eau': 'water',

    # vitamin e
    'tocopherol': 'vitamin e',
    'tocopheryl acetate': 'vitamin e',

    # hyaluronic acid
    'sodium hyaluronate': 'hyaluronic acid',

    # titanium dioxide
    'ci 77891': 'titanium dioxide',
    'titanium dioxide': 'titanium dioxide',
    'titanium dioxide/ci 77891': 'titanium dioxide',
    'ci 77891/titanium dioxide': 'titanium dioxide',

    # iron oxides
    'ci 77491': 'iron oxides',
    'ci 77492': 'iron oxides',
    'ci 77499': 'iron oxides',
    'iron oxides': 'iron oxides',
    'iron oxides/ci 77491': 'iron oxides',
    'iron oxides/ci 77492': 'iron oxides',
    'iron oxides/ci 77499': 'iron oxides',

    # fragrance
    'parfum': 'fragrance',
    'perfume': 'fragrance',
    'fragrance/parfum': 'fragrance',

    # color lakes
    'ci 15850': 'red 7 lake',
    'ci 42090': 'blue 1 lake',

    # oils / botanical ingredients
    'simmondsia chinensis seed oil': 'jojoba oil',
    'helianthus annuus seed oil': 'sunflower oil',
    'zea mays starch': 'corn starch',
    'butyrospermum parkii butter': 'shea butter',

    # keep common cleaned names as-is
    'mica': 'mica',
    'talc': 'talc',
    'glycerin': 'glycerin',
    'butylene glycol': 'butylene glycol',
    'caprylyl glycol': 'caprylyl glycol',
    'phenoxyethanol': 'phenoxyethanol',
    'dimethicone': 'dimethicone'
}

# Apply mapping
ingredients_long['ingredient_clean'] = (
    ingredients_long['ingredient']
    .replace(ingredient_map)
    .str.strip()
)

ingredients_long['ingredient_clean'] = (
    ingredients_long['ingredient_clean']
    .astype(str)
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)     # normalize internal spaces
    .str.replace(r"^\s+|\s+$", "", regex=True) # remove leading/trailing spaces (stronger strip)
    .str.replace(r"^\[+", "", regex=True)
    .str.replace(r"^'+", "", regex=True)
    .str.replace(r"\.+$", "", regex=True)
    .str.replace(r"\)+$", "", regex=True)
    .str.strip()  # final safety pass
)

ingredients_long = ingredients_long[
    ~ingredients_long['ingredient_clean'].isin(['', 'na', '1', 'none', 'nan', '+/-'])
].copy()

# remove duplicate ingredients WITHIN each product only
ingredients_long = ingredients_long.drop_duplicates(
    subset=['product_idx', 'ingredient_clean']
).copy()

# Top 50 ingredients
top_50_ingredients = (
    ingredients_long['ingredient_clean']
    .value_counts()
    .head(50)
    .index
)

ingredients_long_top50 = ingredients_long[
    ingredients_long['ingredient_clean'].isin(top_50_ingredients)
].copy()

ingredients_long_top50.head()

,ingredients,ingredient,product_idx,ingredient_clean
0,"['Polybutene; Hydrogenated Polyisobutene; Dextrin Palmitate; Bis-Diglyceryl Polyacyladipate-2; Polydecene; Magnolia Officinalis Bark Extract; Limnanthes Alba (Meadowfoam) Seed Oil; Lecithin; Caprylyl Glycol; Simethicone; Pentaerythrityl Tetra-Di-T-Butyl Hydroxyhydrocinnamate; Hexylene Glycol; Fragrance (Parfum); Phenoxyethanol; [+/- Titanium Dioxide (Ci 77891); Iron Oxides (Ci 77491, Ci 77492, Ci 77499); Red 7 Lake (Ci 15850); Blue 1 Lake (Ci 42090); Manganese Violet (Ci 77742); Yellow 5 Lake (Ci 19140); Red 22 Lake (Ci 45380); Red 30 Lake (Ci 73360); Mica; Red 6 (Ci 15850); Yellow 6 Lake (Ci 15985); Carmine (Ci 75470); Bismuth Oxychloride (Ci 77163); Red 33 Lake (Ci 17200); Orange 5 (Ci 45370); Red 28 Lake (Ci 45410)].']",ci 77492,0,iron oxides
1,"['Triisostearoyl Polyglyceryl-3 Dimer Dilinoleate, Polybutene, Diisostearyl Malate, Octyldodecanol, Polyglyceryl-2 Triisostearate, VP/Hexadecene Copolymer, Silica Dimethyl', 'Silylate, Adansonia Digitata Seed Oil, Camellia Japonica Seed Oil, Passiflora Incarnata Seed', 'Oil, Pentaclethra Macroloba Seed Oil, Tocopherol, Hexadecene, Pentaerythrityl Tetra-Di-', 'T-Butyl Hydroxyhydrocinnamate, Flavor (Aroma). May Contain: Titanium Dioxide (CI 77891), Iron Oxides (CI 77491, Ci 77492), Blue I Lake (CI 42090), Red 7 Lake (CI 15850), CI 45380 (Red 22 Lake), CI 45410 (Red 28 Lake).']",polybutene,1,polybutene
1,"['Triisostearoyl Polyglyceryl-3 Dimer Dilinoleate, Polybutene, Diisostearyl Malate, Octyldodecanol, Polyglyceryl-2 Triisostearate, VP/Hexadecene Copolymer, Silica Dimethyl', 'Silylate, Adansonia Digitata Seed Oil, Camellia Japonica Seed Oil, Passiflora Incarnata Seed', 'Oil, Pentaclethra Macroloba Seed Oil, Tocopherol, Hexadecene, Pentaerythrityl Tetra-Di-', 'T-Butyl Hydroxyhydrocinnamate, Flavor (Aroma). May Contain: Titanium Dioxide (CI 77891), Iron Oxides (CI 77491, Ci 77492), Blue I Lake (CI 42090), Red 7 Lake (CI 15850), CI 45380 (Red 22 Lake), CI 45410 (Red 28 Lake).']",diisostearyl malate,1,diisostearyl malate
1,"['Triisostearoyl Polyglyceryl-3 Dimer Dilinoleate, Polybutene, Diisostearyl Malate, Octyldodecanol, Polyglyceryl-2 Triisostearate, VP/Hexadecene Copolymer, Silica Dimethyl', 'Silylate, Adansonia Digitata Seed Oil, Camellia Japonica Seed Oil, Passiflora Incarnata Seed', 'Oil, Pentaclethra Macroloba Seed Oil, Tocopherol, Hexadecene, Pentaerythrityl Tetra-Di-', 'T-Butyl Hydroxyhydrocinnamate, Flavor (Aroma). May Contain: Titanium Dioxide (CI 77891), Iron Oxides (CI 77491, Ci 77492), Blue I Lake (CI 42090), Red 7 Lake (CI 15850), CI 45380 (Red 22 Lake), CI 45410 (Red 28 Lake).']",octyldodecanol,1,octyldodecanol
1,"['Triisostearoyl Polyglyceryl-3 Dimer Dilinoleate, Polybutene, Diisostearyl Malate, Octyldodecanol, Polyglyceryl-2 Triisostearate, VP/Hexadecene Copolymer, Silica Dimethyl', 'Silylate, Adansonia Digitata Seed Oil, Camellia Japonica Seed Oil, Passiflora Incarnata Seed', 'Oil, Pentaclethra Macroloba Seed Oil, Tocopherol, Hexadecene, Pentaerythrityl Tetra-Di-', 'T-Butyl Hydroxyhydrocinnamate, Flavor (Aroma). May Contain: Titanium Dioxide (CI 77891), Iron Oxides (CI 77491, Ci 77492), Blue I Lake (CI 42090), Red 7 Lake (CI 15850), CI 45380 (Red 22 Lake), CI 45410 (Red 28 Lake).']",tocopherol,1,vitamin e


In [55]:
# Keep only top 50 ingredients per product
ingredients_clean_per_product = (
    ingredients_long_top50.groupby('product_idx')['ingredient_clean']
    .apply(lambda x: ','.join(x))
)

df_products['ingredients_clean'] = (
    ingredients_clean_per_product
    .reindex(df_products.index)
    .fillna('')
)

vectorizer = TfidfVectorizer(
    token_pattern=r'[^,]+',
    max_features=50,
    min_df=5,
    max_df=0.8
)

X_ingredients = vectorizer.fit_transform(df_products['ingredients_clean'])
feature_names = vectorizer.get_feature_names_out()

ingredients_tfidf_df = pd.DataFrame.sparse.from_spmatrix(
    X_ingredients,
    columns=[f"ing_{col}" for col in feature_names],
    index=df_products.index
)

ingredients_tfidf_df.head()

,ing_2-hexanediol,ing_aluminum hydroxide,ing_bismuth oxychloride,ing_blue 1 lake,ing_butylene glycol,ing_calcium sodium borosilicate,ing_caprylic/capric triglyceride,ing_caprylyl glycol,ing_carmine,ing_citric acid,...,ing_talc,ing_tin oxide,ing_titanium dioxide,ing_triethoxycaprylylsilane,ing_trimethylsiloxysilicate,ing_ultramarines,ing_vitamin e,ing_water,ing_yellow 5 lake,ing_zinc stearate
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0.273008,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0.347636,0.373905,0,0
3,0,0,0,0,0.380642,0,0,0.320292,0,0,...,0,0,0,0,0,0,0.262998,0.282872,0,0
4,0,0,0,0,0,0,0.331628,0,0,0,...,0,0,0,0,0,0,0,0.247807,0,0


Ingredient Analysis & Vectorization (num_ingredients, ingred_...): We transformed the raw, unstructured ingredient lists into numerical features using a combination of string parsing and natural language processing (NLP).

Ingredient Complexity as a Quality Signal: The num_ingredients count serves as a proxy for formula sophistication; in the beauty industry, a higher count or specific "hero" ingredients often correlate with premium pricing and perceived product efficacy.

Enables NLP-Based Modeling: By using CountVectorizer to extract the top 100 most frequent ingredients, we've enabled the model to identify "chemical signatures." This allows the algorithm to detect if specific components—like Hyaluronic Acid or Squalane—are statistically linked to higher customer satisfaction.

Feature Expansion: This step converted a single text column into 101 structured data points, allowing the machine learning model to "read" the label and weigh the impact of formula composition on a product's overall rating.

# Feature Transformation

## 1) Normalize numerical features

In [56]:
# 1. Identify truly continuous/numerical columns
# We filter for columns that have a wide range of values 
# and exclude the 0/1 binary columns explicitly.

cols_to_scale = []
for col in df_products.select_dtypes(include=['number']).columns:
    # Skip IDs
    if col == 'brand_id':
        continue
    
    # Check if the column is binary (only has 2 unique values, e.g., 0 and 1)
    if df_products[col].nunique() <= 2:
        continue
        
    # Optional: Skip columns that start with 'ingred_' if you want 
    # to keep ingredient counts as-is
    if col.startswith('ingred_'):
        continue
        
    cols_to_scale.append(col)

print("Refined list of columns to scale:")
print(cols_to_scale)

# 1. Select the numerical columns that need scaling
# We exclude binary/dummy columns (0/1) as they are already bounded
cols_to_scale = [
     'price_log', 'discount_pct', 'value_ratio','log_loves',
   'child_count', 'child_max_price', 'child_min_price', 
]

# 2. Initialize and apply the StandardScaler
scaler = StandardScaler()
df_products[cols_to_scale] = scaler.fit_transform(df_products[cols_to_scale])

# 3. Verify the transformation (Means should be approx 0 and Std should be 1)
df_products[cols_to_scale].describe().round(2).T

Refined list of columns to scale:
['loves_count', 'rating', 'reviews', 'price_usd', 'value_price_usd', 'sale_price_usd', 'child_count', 'child_max_price', 'child_min_price', 'price_log', 'discount_pct', 'value_ratio', 'log_loves', 'loves_per_price', 'popularity_index']


,count,mean,std,min,25%,50%,75%,max
price_log,2369.0,0.0,1.0,-4.38,-0.59,-0.02,0.61,5.11
discount_pct,2369.0,0.0,1.0,-0.25,-0.25,-0.25,-0.25,6.82
value_ratio,2369.0,-0.0,1.0,-5.70,-0.16,-0.16,-0.16,16.21
log_loves,2369.0,0.0,1.0,-6.28,-0.58,0.05,0.60,2.67
child_count,2369.0,-0.0,1.0,-0.51,-0.51,-0.40,0.03,10.86
child_max_price,2369.0,0.0,1.0,-0.90,-0.90,-0.09,0.66,10.15
child_min_price,2369.0,0.0,1.0,-0.88,-0.88,-0.15,0.64,10.64


Feature Standardization (StandardScaler): we applied $z$-score normalization to all continuous numerical variables (e.g., price_usd, loves_count, num_ingredients) to ensure they share a common scale with a mean of 0 and a standard deviation of 1.

Ensures Model Fairness: machine learning algorithms (like Linear Regression or KNN) often misinterpret large raw numbers—such as a loves_count of 1,000,000—as being mathematically more "important" than a rating of 4.5. Scaling prevents this bias by putting all features on a level playing field.

Improves Convergence: for gradient-based models (like Neural Networks or XGBoost), having features on the same scale allows the model to find the optimal solution much faster and more reliably.

Handles Distribution Variance: by standardizing engineered features like popularity_index and loves_per_price, we make the relative "success" of a product comparable across the entire dataset, regardless of the original unit of measurement.

## 2) Category features transformation

In [57]:
# Get a list of all categorical column names
categorical_cols_list = df_products.select_dtypes(include=['object', 'category']).columns.tolist()

# Print the count and the names
print(f"Total categorical columns: {len(categorical_cols_list)}")
print(categorical_cols_list)

# 1. Define all categorical features that provide structural meaning
categorical_cols = [ 'secondary_category','tertiary_category', 
                    'price_bucket', 'brand_group'
]

# 2. Apply One-Hot Encoding across all selected features
# We include brand_group and price_bucket to capture brand equity and market tier signals
df_products = pd.get_dummies(
    df_products, 
    columns=categorical_cols, 
    drop_first=True,
    dtype=int
)

Total categorical columns: 15
['product_id', 'product_name', 'brand_name', 'size', 'variation_type', 'variation_value', 'variation_desc', 'ingredients', 'highlights', 'primary_category', 'secondary_category', 'tertiary_category', 'price_bucket', 'brand_group', 'ingredients_clean']


Categorical Encoding like secondary_category: We converted the text-based category labels into a series of binary (0 or 1) columns using One-Hot Encoding.

Category-Specific Success Patterns: Product categories strongly influence performance metrics; for example, Skincare products often have higher price points and repeat purchase rates compared to Makeup, which is more trend-driven.

Eliminating the "Dummy Variable Trap": By using drop_first=True, we ensured mathematical independence between our new columns, preventing multicollinearity which can skew model coefficients.

Machine Learning Readiness: Since algorithms cannot "read" text, this transformation translates the product's department into a numerical format that allows the model to calculate the specific "weight" or impact of being in a high-growth category like Fragrance versus Bath & Body.

## 3) Reviews Dataset

In [58]:
# Text normalization (download below if needed)
# nltk.download('stopwords')
wpt = nltk.WordPunctTokenizer()
stop_words = nltk.corpus.stopwords.words('english')

def normalize_document(doc):
    doc = str(doc)
    doc = re.sub(r'[^a-zA-Z0-9\s]', '', doc, flags=re.I)  # remove special chars
    doc = doc.lower()
    doc = doc.strip()
    tokens = wpt.tokenize(doc)
    filtered_tokens = [token for token in tokens if token not in stop_words]
    doc = ' '.join(filtered_tokens)
    return doc

# Clean review text
df_reviews_sample = df_reviews.sample(10000, random_state=42).copy()

df_reviews_sample['review_text_clean'] = (
    df_reviews_sample['review_text']
    .fillna('')
    .apply(normalize_document)
)

In [59]:
# TF-IDF on review_text_only
vectorizer_reviews = TfidfVectorizer(
    max_features=3000,
    min_df=5,
    max_df=0.8,
    ngram_range=(1, 2)
)

X_reviews = vectorizer_reviews.fit_transform(df_reviews_sample['review_text_clean'])
review_features = vectorizer_reviews.get_feature_names_out()

reviews_tfidf_df = pd.DataFrame.sparse.from_spmatrix(
    X_reviews,
    columns=[f"rev_{col}" for col in review_features],
    index=df_reviews_sample.index
)

# keep identifiers
reviews_tfidf_df = pd.concat(
    [
        df_reviews_sample[['rating']].reset_index(drop=True),
        reviews_tfidf_df.reset_index(drop=True)
    ],
    axis=1
)

reviews_tfidf_df.head()

,rating,rev_10,rev_10 minutes,rev_10 years,rev_100,rev_1010,rev_12,rev_15,rev_20,rev_20 minutes,...,rev_youre,rev_youre looking,rev_youth,rev_youthful,rev_youtube,rev_youve,rev_yttp,rev_zero,rev_zits,rev_zone
0,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,5,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,5,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,5,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,5,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## 4) VADER Sentiment for Reviews

In [60]:
df_reviews_sample = df_reviews_sample[
    ['rating', 'review_text']
]

analyzer = SentimentIntensityAnalyzer()

def get_vader_scores(text):
    text = str(text).strip()
    
    if not text:
        return pd.Series({
            'negative_score': None,
            'neutral_score': None,
            'positive_score': None,
            'compound_score': None
        })
    
    scores = analyzer.polarity_scores(text)
    
    return pd.Series({
        'negative_score': scores['neg'],
        'neutral_score': scores['neu'],
        'positive_score': scores['pos'],
        'compound_score': scores['compound']
    })

vader_results = df_reviews_sample['review_text'].apply(get_vader_scores)

df_reviews_vader = pd.concat(
    [df_reviews_sample, vader_results],
    axis=1
)

pd.set_option('display.max_colwidth', None)
print(df_reviews_vader.head(10))
print(df_reviews_vader.shape)

         rating  \
1049894       1   
619554        5   
634782        5   
706815        5   
731656        5   
37430         5   
845174        5   
570505        5   
347448        3   
672831        5   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                      

## Final Review Dataset

In [61]:
sentiment_df = df_reviews_vader[
    ['compound_score']
].reset_index(drop=True)

reviews_tfidf_df = pd.concat(
    [reviews_tfidf_df.reset_index(drop=True), sentiment_df],
    axis=1
)

scaler = StandardScaler()

reviews_tfidf_df[['compound_score']] = scaler.fit_transform(
    reviews_tfidf_df[['compound_score']]
)

# Use the below for modeling with output = "rating"
reviews_tfidf_df.head()

,rating,rev_10,rev_10 minutes,rev_10 years,rev_100,rev_1010,rev_12,rev_15,rev_20,rev_20 minutes,...,rev_youre looking,rev_youth,rev_youthful,rev_youtube,rev_youve,rev_yttp,rev_zero,rev_zits,rev_zone,compound_score
0,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,-0.830245
1,5,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0.593412
2,5,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0.108302
3,5,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0.596508
4,5,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0.595317


## Final Product Dataset

In [62]:
# We drop raw text and ID columns to keep the dataframe strictly numeric/encoded
cols_to_drop = [
    'product_id', 'product_name', 'brand_id', 'brand_name',
     'price_usd', 'value_price_usd', 'sale_price_usd', 'loves_count', 
    'rating', 'reviews', 'log_loves', 'loves_per_price', 'popularity_index',
    'size', 'variation_type', 'variation_value', 
    'variation_desc', 'ingredients', 'ingredients_clean', 'highlights' 
]

df_product_final = df_products.drop(columns=cols_to_drop)
df_product_final = pd.concat(
    [df_product_final, ingredients_tfidf_df],
    axis=1
)

# Verify the final shape (should be significantly wider now)
print(f"Final Product Features: {df_product_final.shape[1]}")
df_product_final.head()

Final Product Features: 136


,limited_edition,new,online_only,out_of_stock,sephora_exclusive,primary_category,child_count,child_max_price,child_min_price,price_log,...,ing_talc,ing_tin oxide,ing_titanium dioxide,ing_triethoxycaprylylsilane,ing_trimethylsiloxysilicate,ing_ultramarines,ing_vitamin e,ing_water,ing_yellow 5 lake,ing_zinc stearate
0,0,0,1,0,0,Makeup,-0.512608,-0.900507,-0.877163,0.379117,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,1,Makeup,-0.187795,0.177735,0.246876,-0.787481,...,0,0,0,0,0,0,0.273008,0,0,0
2,0,0,0,0,1,Makeup,0.028747,0.824680,0.921300,0.190790,...,0,0,0,0,0,0,0.347636,0.373905,0,0
3,0,0,0,0,1,Makeup,-0.512608,-0.900507,-0.877163,-0.893082,...,0,0,0,0,0,0,0.262998,0.282872,0,0
4,0,0,1,0,1,Makeup,-0.296066,0.285559,0.359280,-0.590583,...,0,0,0,0,0,0,0,0.247807,0,0


In [63]:
## Review below

In [64]:
list(df_product_final.columns)

['limited_edition',
 'new',
 'online_only',
 'out_of_stock',
 'sephora_exclusive',
 'primary_category',
 'child_count',
 'child_max_price',
 'child_min_price',
 'price_log',
 'is_on_sale',
 'discount_pct',
 'value_ratio',
 'secondary_category_Brushes & Applicators',
 'secondary_category_Cheek',
 'secondary_category_Eye',
 'secondary_category_Face',
 'secondary_category_Lip',
 'secondary_category_Makeup Palettes',
 'secondary_category_Mini Size',
 'secondary_category_Nail',
 'secondary_category_Value & Gift Sets',
 'tertiary_category_Blotting Papers',
 'tertiary_category_Blush',
 'tertiary_category_Bronzer',
 'tertiary_category_Brush Cleaners',
 'tertiary_category_Brush Sets',
 'tertiary_category_Cheek Palettes',
 'tertiary_category_Color Correct',
 'tertiary_category_Concealer',
 'tertiary_category_Contour',
 'tertiary_category_Eye Brushes',
 'tertiary_category_Eye Palettes',
 'tertiary_category_Eye Primer',
 'tertiary_category_Eye Sets',
 'tertiary_category_Eyebrow',
 'tertiary_catego

# Data Pipeline

## Product File Data Pipeline

In [199]:
# Define the S3 path
s3_path = 's3://ads508-team1-sephora/curated/products/products.parquet'

# Load the parquet directly into a DataFrame
df_raw = pd.read_parquet(s3_path)


def run_product_pipeline(df_raw, top_n_secondary=5, n_svd_components=10):
    # 1. INITIAL FILTERING
    df = (
        df_raw[df_raw['primary_category'] == 'Makeup']
        .copy()
        .reset_index(drop=True)
    )

    # 2. IMPUTATION
    df['rating'] = pd.to_numeric(df['rating'], errors='coerce')
    df['reviews'] = pd.to_numeric(df['reviews'], errors='coerce').fillna(0)
    df['loves_count'] = pd.to_numeric(df['loves_count'], errors='coerce').fillna(0)

    cols_to_fix = [
        'size', 'variation_type', 'variation_value', 'variation_desc',
        'ingredients', 'highlights', 'secondary_category', 'tertiary_category'
    ]
    for col in cols_to_fix:
        df[col] = df[col].fillna("NA")

    df['price_usd'] = pd.to_numeric(df['price_usd'], errors='coerce')
    df['value_price_usd'] = pd.to_numeric(df['value_price_usd'], errors='coerce')
    df['sale_price_usd'] = pd.to_numeric(df['sale_price_usd'], errors='coerce')
    df['child_max_price'] = pd.to_numeric(df['child_max_price'], errors='coerce')
    df['child_min_price'] = pd.to_numeric(df['child_min_price'], errors='coerce')

    df['price_usd'] = df['price_usd'].fillna(df['price_usd'].median())
    df['value_price_usd'] = df['value_price_usd'].fillna(df['price_usd'])
    df['sale_price_usd'] = df['sale_price_usd'].fillna(df['price_usd'])
    df['child_max_price'] = df['child_max_price'].fillna(0)
    df['child_min_price'] = df['child_min_price'].fillna(0)

    bool_cols = ['limited_edition', 'new', 'online_only', 'out_of_stock', 'sephora_exclusive']
    for col in bool_cols:
        if col in df.columns:
            df[col] = df[col].fillna(False).astype(int)
        else:
            df[col] = 0

    # 3. CORE NUMERICAL FEATURES
    df['price_log'] = np.log1p(df['price_usd'].clip(lower=0))
    df['log_loves'] = np.log1p(df['loves_count'].clip(lower=0))
    df['reviews_log'] = np.log1p(df['reviews'].clip(lower=0))

    df['num_ingredients'] = (
        df['ingredients']
        .fillna('')
        .apply(lambda x: len([i for i in str(x).split(',') if str(i).strip() not in ['', 'NA']]))
    )

    df['price_x_num_ing'] = df['price_log'] * df['num_ingredients']
    df['price_x_reviews'] = df['price_log'] * df['reviews_log']

    # 4. LIMIT SECONDARY CATEGORY TO TOP N
    top_secondary = df['secondary_category'].value_counts().nlargest(top_n_secondary).index
    df['secondary_category_top'] = df['secondary_category'].apply(
        lambda x: x if x in top_secondary else 'Other'
    )

    # 5. INGREDIENT FEATURE ENGINEERING
    ingredients_long = (
        df[['ingredients']]
        .copy()
        .assign(product_idx=lambda x: x.index)
        .assign(ingredient=lambda x: x['ingredients'].fillna('NA').astype(str).str.split(','))
        .explode('ingredient')
        .copy()
    )

    ingredients_long['ingredient'] = (
        ingredients_long['ingredient']
        .astype(str)
        .str.lower()
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
        .str.replace(r"^\[+", "", regex=True)
        .str.replace(r"^'+", "", regex=True)
        .str.replace(r"^\[\s*\+/-\s*", "", regex=True)
        .str.replace(r"^\+/-\s*", "", regex=True)
        .str.replace(r"\.+$", "", regex=True)
        .str.replace(r"\)+$", "", regex=True)
        .str.replace(r"\(.*", "", regex=True)
        .str.replace(r"\s*[\\/]\s*", "/", regex=True)
        .str.strip()
    )

    ingredients_long = ingredients_long[
        ~ingredients_long['ingredient'].isin(['', 'na', '1', 'none', 'nan', '+/-'])
    ].copy()

    ingredient_map = {
        'aqua': 'water',
        'eau': 'water',
        'water': 'water',
        'water/aqua': 'water',
        'aqua/water': 'water',
        'aqua/water/eau': 'water',
        'water/aqua/eau': 'water',
        'water//aqua//eau': 'water',
        'water\\aqua\\eau': 'water',
        'tocopherol': 'vitamin e',
        'tocopheryl acetate': 'vitamin e',
        'sodium hyaluronate': 'hyaluronic acid',
        'ci 77891': 'titanium dioxide',
        'titanium dioxide': 'titanium dioxide',
        'titanium dioxide/ci 77891': 'titanium dioxide',
        'ci 77891/titanium dioxide': 'titanium dioxide',
        'ci 77491': 'iron oxides',
        'ci 77492': 'iron oxides',
        'ci 77499': 'iron oxides',
        'iron oxides': 'iron oxides',
        'iron oxides/ci 77491': 'iron oxides',
        'iron oxides/ci 77492': 'iron oxides',
        'iron oxides/ci 77499': 'iron oxides',
        'parfum': 'fragrance',
        'perfume': 'fragrance',
        'fragrance/parfum': 'fragrance',
        'ci 15850': 'red 7 lake',
        'ci 42090': 'blue 1 lake',
        'simmondsia chinensis seed oil': 'jojoba oil',
        'helianthus annuus seed oil': 'sunflower oil',
        'zea mays starch': 'corn starch',
        'butyrospermum parkii butter': 'shea butter',
        'mica': 'mica',
        'talc': 'talc',
        'glycerin': 'glycerin',
        'butylene glycol': 'butylene glycol',
        'caprylyl glycol': 'caprylyl glycol',
        'phenoxyethanol': 'phenoxyethanol',
        'dimethicone': 'dimethicone'
    }

    ingredients_long['ingredient_clean'] = (
        ingredients_long['ingredient']
        .replace(ingredient_map)
        .astype(str)
        .str.lower()
        .str.replace(r"\s+", " ", regex=True)
        .str.replace(r"^\s+|\s+$", "", regex=True)
        .str.replace(r"^\[+", "", regex=True)
        .str.replace(r"^'+", "", regex=True)
        .str.replace(r"\.+$", "", regex=True)
        .str.replace(r"\)+$", "", regex=True)
        .str.strip()
    )

    ingredients_long = ingredients_long[
        ~ingredients_long['ingredient_clean'].isin(['', 'na', '1', 'none', 'nan', '+/-'])
    ].copy()

    ingredients_long = ingredients_long.drop_duplicates(
        subset=['product_idx', 'ingredient_clean']
    ).copy()

    # 6. SPLIT FIRST
    base_cols = [
        'reviews_log',
        'limited_edition',
        'online_only',
        'sephora_exclusive',
        'price_log',
        'num_ingredients',
        'price_x_num_ing',
        'price_x_reviews',
        'secondary_category_top'
    ]

    valid_rating_idx = df[df['rating'].notna()].index

    train_idx, temp_idx = train_test_split(valid_rating_idx, test_size=0.10, random_state=42)
    val_idx, test_idx = train_test_split(temp_idx, test_size=0.50, random_state=42)

    df_train = df.loc[train_idx].copy()
    df_val = df.loc[val_idx].copy()
    df_test = df.loc[test_idx].copy()

    # 7. TOP INGREDIENTS FROM TRAIN ONLY
    top_50_ingredients = (
        ingredients_long.loc[ingredients_long['product_idx'].isin(train_idx), 'ingredient_clean']
        .value_counts()
        .head(50)
        .index
    )

    ingredients_long_top50 = ingredients_long[
        ingredients_long['ingredient_clean'].isin(top_50_ingredients)
    ].copy()

    def build_ingredient_text(df_subset):
        subset_long = (
            df_subset[['ingredients']]
            .copy()
            .assign(product_idx=lambda x: x.index)
            .assign(ingredient=lambda x: x['ingredients'].fillna('NA').astype(str).str.split(','))
            .explode('ingredient')
        )

        subset_long['ingredient'] = (
            subset_long['ingredient']
            .astype(str)
            .str.lower()
            .str.strip()
            .str.replace(r"\s+", " ", regex=True)
            .str.replace(r"^\[+", "", regex=True)
            .str.replace(r"^'+", "", regex=True)
            .str.replace(r"^\[\s*\+/-\s*", "", regex=True)
            .str.replace(r"^\+/-\s*", "", regex=True)
            .str.replace(r"\.+$", "", regex=True)
            .str.replace(r"\)+$", "", regex=True)
            .str.replace(r"\(.*", "", regex=True)
            .str.replace(r"\s*[\\/]\s*", "/", regex=True)
            .str.strip()
        )

        subset_long['ingredient_clean'] = (
            subset_long['ingredient']
            .replace(ingredient_map)
            .astype(str)
            .str.lower()
            .str.replace(r"\s+", " ", regex=True)
            .str.replace(r"^\s+|\s+$", "", regex=True)
            .str.replace(r"^\[+", "", regex=True)
            .str.replace(r"^'+", "", regex=True)
            .str.replace(r"\.+$", "", regex=True)
            .str.replace(r"\)+$", "", regex=True)
            .str.strip()
        )

        subset_long = subset_long[
            subset_long['ingredient_clean'].isin(top_50_ingredients)
        ].drop_duplicates(subset=['product_idx', 'ingredient_clean'])

        text = (
            subset_long.groupby('product_idx')['ingredient_clean']
            .apply(lambda x: ','.join(x))
        )

        return text.reindex(df_subset.index).fillna('')

    df_train['ingredients_clean'] = build_ingredient_text(df_train)
    df_val['ingredients_clean'] = build_ingredient_text(df_val)
    df_test['ingredients_clean'] = build_ingredient_text(df_test)

    # 8. EXPLICIT INGREDIENT FLAGS
    for subset in [df_train, df_val, df_test]:
        subset['has_fragrance'] = subset['ingredients_clean'].str.contains('fragrance', regex=False).astype(int)
        subset['has_hyaluronic_acid'] = subset['ingredients_clean'].str.contains('hyaluronic acid', regex=False).astype(int)

    # NEW: extra top ingredient flags
    existing_manual = {'fragrance', 'hyaluronic acid'}

    weak_ingredients = {
        'iron oxides',
        'titanium dioxide',
        'water',
        'vitamin e',
        'silica',
        'mica'
    }

    borderline_ingredients = {
        'phenoxyethanol',
        'caprylyl glycol',
        'butylene glycol'
    }

    top_flag_ingredients = [
        ing for ing in list(top_50_ingredients[:30])
        if ing not in existing_manual
        and ing not in weak_ingredients
        and ing not in borderline_ingredients
    ][:10]

    def sanitize_feature_name(text):
        return (
            str(text)
            .lower()
            .replace(' ', '_')
            .replace('/', '_')
            .replace('-', '_')
            .replace('.', '')
            .replace('(', '')
            .replace(')', '')
        )

    top_flag_feature_names = {
        ing: f"has_{sanitize_feature_name(ing)}"
        for ing in top_flag_ingredients
    }

    def add_top_ingredient_flags(df_subset):
        for ing in top_flag_ingredients:
            feature_name = top_flag_feature_names[ing]
            df_subset[feature_name] = df_subset['ingredients_clean'].str.contains(ing, regex=False).astype(int)
        return df_subset

    df_train = add_top_ingredient_flags(df_train)
    df_val = add_top_ingredient_flags(df_val)
    df_test = add_top_ingredient_flags(df_test)

    # 9. TF-IDF + SVD FIT ON TRAIN ONLY
    vectorizer = TfidfVectorizer(
        token_pattern=r'[^,]+',
        max_features=50,
        min_df=2,
        max_df=0.95
    )

    X_train_ing = vectorizer.fit_transform(df_train['ingredients_clean'])
    X_val_ing = vectorizer.transform(df_val['ingredients_clean'])
    X_test_ing = vectorizer.transform(df_test['ingredients_clean'])

    max_possible_components = min(
        n_svd_components,
        max(1, X_train_ing.shape[0] - 1),
        max(1, X_train_ing.shape[1] - 1)
    )

    svd = TruncatedSVD(n_components=max_possible_components, random_state=42)
    X_train_ing_svd = svd.fit_transform(X_train_ing)
    X_val_ing_svd = svd.transform(X_val_ing)
    X_test_ing_svd = svd.transform(X_test_ing)

    ingredients_svd_train = pd.DataFrame(
        X_train_ing_svd,
        columns=[f"ing_svd_{i+1}" for i in range(max_possible_components)],
        index=df_train.index
    )

    ingredients_svd_val = pd.DataFrame(
        X_val_ing_svd,
        columns=[f"ing_svd_{i+1}" for i in range(max_possible_components)],
        index=df_val.index
    )

    ingredients_svd_test = pd.DataFrame(
        X_test_ing_svd,
        columns=[f"ing_svd_{i+1}" for i in range(max_possible_components)],
        index=df_test.index
    )

    for i in range(max_possible_components + 1, n_svd_components + 1):
        ingredients_svd_train[f"ing_svd_{i}"] = 0.0
        ingredients_svd_val[f"ing_svd_{i}"] = 0.0
        ingredients_svd_test[f"ing_svd_{i}"] = 0.0

    ingredients_svd_train = ingredients_svd_train[[f"ing_svd_{i+1}" for i in range(n_svd_components)]]
    ingredients_svd_val = ingredients_svd_val[[f"ing_svd_{i+1}" for i in range(n_svd_components)]]
    ingredients_svd_test = ingredients_svd_test[[f"ing_svd_{i+1}" for i in range(n_svd_components)]]

    explained_var = svd.explained_variance_ratio_.sum()
    print(f"Ingredient SVD explained variance: {explained_var:.4f}")

    # 10. ENCODING + SCALING FIT ON TRAIN ONLY
    top_flag_cols = [top_flag_feature_names[ing] for ing in top_flag_ingredients]

    X_train_tab = pd.get_dummies(
        df_train[base_cols + ['has_fragrance', 'has_hyaluronic_acid'] + top_flag_cols],
        columns=['secondary_category_top'],
        drop_first=True,
        dtype=int
    )
    X_val_tab = pd.get_dummies(
        df_val[base_cols + ['has_fragrance', 'has_hyaluronic_acid'] + top_flag_cols],
        columns=['secondary_category_top'],
        drop_first=True,
        dtype=int
    )
    X_test_tab = pd.get_dummies(
        df_test[base_cols + ['has_fragrance', 'has_hyaluronic_acid'] + top_flag_cols],
        columns=['secondary_category_top'],
        drop_first=True,
        dtype=int
    )

    X_val_tab = X_val_tab.reindex(columns=X_train_tab.columns, fill_value=0)
    X_test_tab = X_test_tab.reindex(columns=X_train_tab.columns, fill_value=0)

    cols_to_scale = [
        'price_log',
        'reviews_log',
        'num_ingredients',
        'price_x_num_ing',
        'price_x_reviews'
    ]
    scaler = StandardScaler()

    X_train_tab[cols_to_scale] = scaler.fit_transform(X_train_tab[cols_to_scale])
    X_val_tab[cols_to_scale] = scaler.transform(X_val_tab[cols_to_scale])
    X_test_tab[cols_to_scale] = scaler.transform(X_test_tab[cols_to_scale])

    # 11. FINAL MATRICES
    X_train = pd.concat([X_train_tab, ingredients_svd_train], axis=1)
    X_val = pd.concat([X_val_tab, ingredients_svd_val], axis=1)
    X_test = pd.concat([X_test_tab, ingredients_svd_test], axis=1)

    X_train = X_train.loc[:, ~X_train.columns.duplicated()].copy()
    X_val = X_val.loc[:, ~X_val.columns.duplicated()].copy()
    X_test = X_test.loc[:, ~X_test.columns.duplicated()].copy()

    y_train_r = df.loc[train_idx, 'rating']
    y_val_r = df.loc[val_idx, 'rating']
    y_test_r = df.loc[test_idx, 'rating']

    y_train_l = df.loc[train_idx, 'log_loves']
    y_val_l = df.loc[val_idx, 'log_loves']
    y_test_l = df.loc[test_idx, 'log_loves']

    print(f"Pipeline Complete. Training features: {X_train.shape[1]}")
    print(f"Rating Training Samples: {len(X_train)}")
    print(f"Love Training Samples: {len(X_train)}")
    print("Final feature columns:")
    print(X_train.columns.tolist())

    return (
        X_train, X_val, X_test,
        y_train_r, y_val_r, y_test_r,
        y_train_l, y_val_l, y_test_l,
        ingredients_long, ingredients_long_top50,
        vectorizer, svd
    )

(
    X_train_r, X_val_r, X_test_r,
    y_train_r, y_val_r, y_test_r,
    y_train_l, y_val_l, y_test_l,
    ingredients_long, ingredients_long_top50,
    vectorizer, svd
) = run_product_pipeline(df_raw, top_n_secondary=5, n_svd_components=10)

Ingredient SVD explained variance: 0.4326
Pipeline Complete. Training features: 35
Rating Training Samples: 2095
Love Training Samples: 2095
Final feature columns:
['reviews_log', 'limited_edition', 'online_only', 'sephora_exclusive', 'price_log', 'num_ingredients', 'price_x_num_ing', 'price_x_reviews', 'has_fragrance', 'has_hyaluronic_acid', 'has_dimethicone', 'has_glycerin', 'has_caprylic_capric_triglyceride', 'has_synthetic_fluorphlogopite', 'has_tin_oxide', 'has_ethylhexylglycerin', 'has_blue_1_lake', 'has_polyethylene', 'has_yellow_5_lake', 'has_pentaerythrityl_tetra_di_t_butyl_hydroxyhydrocinnamate', 'secondary_category_top_Cheek', 'secondary_category_top_Eye', 'secondary_category_top_Face', 'secondary_category_top_Lip', 'secondary_category_top_Other', 'ing_svd_1', 'ing_svd_2', 'ing_svd_3', 'ing_svd_4', 'ing_svd_5', 'ing_svd_6', 'ing_svd_7', 'ing_svd_8', 'ing_svd_9', 'ing_svd_10']


# Save The Final Splits into Model folder

In [200]:
import os
import boto3
import pandas as pd
import shutil

# 1. Define your S3 bucket and folder path
bucket_name = 'ads508-team1-sephora'
s3_model_path = 'Model/final_splits'

# 2. Local directory to temporarily store files before upload
local_dir = './temp_splits'
os.makedirs(local_dir, exist_ok=True)

# 3. Create a helper dictionary for all your datasets
# Note: Ensure these variables (X_train_r, etc.) exist in your memory
data_to_save = {
    "X_train_rating": pd.DataFrame(X_train_r), 
    "X_val_rating": pd.DataFrame(X_val_r), 
    "X_test_rating": pd.DataFrame(X_test_r),
    "y_train_rating": pd.DataFrame(y_train_r), 
    "y_val_rating": pd.DataFrame(y_val_r), 
    "y_test_rating": pd.DataFrame(y_test_r),
    
    "X_train_love": pd.DataFrame(X_train_l), 
    "X_val_love": pd.DataFrame(X_val_l), 
    "X_test_love": pd.DataFrame(X_test_l),
    "y_train_love": pd.DataFrame(y_train_l), 
    "y_val_love": pd.DataFrame(y_val_l), 
    "y_test_love": pd.DataFrame(y_test_l)
}

# 4. Save locally then upload to S3
s3_client = boto3.client('s3')

for name, df in data_to_save.items():
    local_file = f"{local_dir}/{name}.parquet"
    
    df = df.loc[:, ~df.columns.duplicated()]
    
    # Force all sparse columns to dense
    for col in df.columns:
        if pd.api.types.is_sparse(df[col]):
            df[col] = df[col].sparse.to_dense()
    
    df.to_parquet(local_file, index=False)
    
    s3_key = f"{s3_model_path}/{name}.parquet"
    s3_client.upload_file(local_file, bucket_name, s3_key)
    print(f"Successfully uploaded: s3://{bucket_name}/{s3_key}")

# 5. Clean up local files
shutil.rmtree(local_dir)

/tmp/ipykernel_1399/2927059574.py:42: DeprecationWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if pd.api.types.is_sparse(df[col]):


Successfully uploaded: s3://ads508-team1-sephora/Model/final_splits/X_train_rating.parquet
Successfully uploaded: s3://ads508-team1-sephora/Model/final_splits/X_val_rating.parquet
Successfully uploaded: s3://ads508-team1-sephora/Model/final_splits/X_test_rating.parquet
Successfully uploaded: s3://ads508-team1-sephora/Model/final_splits/y_train_rating.parquet
Successfully uploaded: s3://ads508-team1-sephora/Model/final_splits/y_val_rating.parquet
Successfully uploaded: s3://ads508-team1-sephora/Model/final_splits/y_test_rating.parquet
Successfully uploaded: s3://ads508-team1-sephora/Model/final_splits/X_train_love.parquet
Successfully uploaded: s3://ads508-team1-sephora/Model/final_splits/X_val_love.parquet
Successfully uploaded: s3://ads508-team1-sephora/Model/final_splits/X_test_love.parquet
Successfully uploaded: s3://ads508-team1-sephora/Model/final_splits/y_train_love.parquet
Successfully uploaded: s3://ads508-team1-sephora/Model/final_splits/y_val_love.parquet
Successfully uploade